# Script 4 — Avaliação nos Dados de Teste
**Relatório de métricas por algoritmo, target e setor**

In [ ]:

import pandas as pd, numpy as np, pickle, warnings
from pathlib import Path
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
PASTA_SAIDA = Path('outputs')
print("✅ Dependências carregadas")


In [ ]:

teste  = pd.read_parquet(PASTA_SAIDA / 'teste.parquet')
treino = pd.read_parquet(PASTA_SAIDA / 'treino.parquet')
with open(PASTA_SAIDA / 'features.pkl','rb') as f: FEATURES = pickle.load(f)
with open(PASTA_SAIDA / 'targets.pkl','rb') as f:  TARGETS  = pickle.load(f)
with open(PASTA_SAIDA / 'resultados_cv.pkl','rb') as f: resultados = pickle.load(f)
print(f"Teste: {teste.shape} | Targets: {TARGETS}")


## Métricas no conjunto de teste

In [ ]:

def metricas_teste(y_true, y_pred):
    rmse = np.sqrt(np.mean((y_true - y_pred)**2))
    mae  = np.mean(np.abs(y_true - y_pred))
    mask = y_true != 0
    mape = np.mean(np.abs((y_true[mask]-y_pred[mask])/y_true[mask])) if mask.sum()>0 else np.nan
    ss_res = np.sum((y_true - y_pred)**2)
    ss_tot = np.sum((y_true - y_true.mean())**2)
    r2 = 1 - ss_res/ss_tot if ss_tot != 0 else np.nan
    return {'RMSE':rmse,'MAE':mae,'MAPE':mape,'R2':r2}

rows_teste = []
melhores_modelos = {}

for target in TARGETS:
    df_t = teste[FEATURES + [target] + ['SETOR','NOME_CIA']].dropna(subset=[target])
    X_test = df_t[FEATURES].values
    y_test = df_t[target].values
    melhor_mape = np.inf
    melhor_nome = None

    for alg in resultados.get(target, {}):
        caminho = PASTA_SAIDA / f'modelo_{target}_{alg}.pkl'
        if not caminho.exists():
            continue
        modelo = joblib.load(caminho)
        y_pred = modelo.predict(X_test)
        m = metricas_teste(y_test, y_pred)
        rows_teste.append({'Target':target,'Algoritmo':alg,**m})
        print(f"{target} | {alg:<20} RMSE={m['RMSE']:,.0f} MAPE={m['MAPE']:.2%} R²={m['R2']:.3f}")
        if m['MAPE'] < melhor_mape:
            melhor_mape = m['MAPE']
            melhor_nome = alg
            melhores_modelos[target] = (alg, modelo)

    if melhor_nome:
        print(f"  ✅ Melhor para {target}: {melhor_nome} (MAPE={melhor_mape:.2%})")

df_teste = pd.DataFrame(rows_teste)
df_teste.to_csv(PASTA_SAIDA / 'resultados_teste.csv', index=False)


## Análise por setor

In [ ]:

for target in TARGETS:
    if target not in melhores_modelos:
        continue
    alg, modelo = melhores_modelos[target]
    df_t = teste[FEATURES + [target,'SETOR']].dropna(subset=[target])
    X = df_t[FEATURES].values
    y = df_t[target].values
    df_t = df_t.copy()
    df_t['y_pred'] = modelo.predict(X)
    df_t['erro_abs'] = np.abs(df_t[target] - df_t['y_pred'])

    print(f"\n{target} — {alg} — por setor:")
    setor_res = df_t.groupby('SETOR').apply(
        lambda g: pd.Series(metricas_teste(g[target].values, g['y_pred'].values))
    )
    print(setor_res[['MAPE','R2']].round(3).to_string())


## Feature Importance (Random Forest)

In [ ]:

for target in TARGETS:
    caminho_rf = PASTA_SAIDA / f'modelo_{target}_RandomForest.pkl'
    if not caminho_rf.exists():
        continue
    modelo_rf = joblib.load(caminho_rf)
    importancias = pd.Series(modelo_rf.feature_importances_, index=FEATURES)
    importancias = importancias.sort_values(ascending=False).head(10)
    print(f"\nTop-10 features — {target}:")
    print(importancias.to_string())
    fig, ax = plt.subplots(figsize=(8,4))
    importancias.plot(kind='barh', ax=ax, color='steelblue')
    ax.invert_yaxis()
    ax.set_title(f'Feature Importance — Random Forest | {target}')
    plt.tight_layout()
    plt.savefig(PASTA_SAIDA / f'feat_importance_{target}.png', dpi=150)
    plt.show()


In [ ]:

import pickle
with open(PASTA_SAIDA / 'melhores_modelos.pkl', 'wb') as f:
    pickle.dump({k: (nome,) for k, (nome, _) in melhores_modelos.items()}, f)
# Salva modelos melhores separados
for target, (alg, modelo) in melhores_modelos.items():
    import joblib
    joblib.dump(modelo, PASTA_SAIDA / f'melhor_modelo_{target}.pkl')
print("✅ Relatório de avaliação concluído e modelos melhores salvos")
